# Splice-site dataset validation

Checks whether the generated dataset is ready for modeling.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# This notebook is expected to be in splice-finder/notebooks/.
PROJECT_DIR = Path.cwd().parent
DATA_CSV = PROJECT_DIR / "data" / "processed" / "splice_dataset.csv"

if not DATA_CSV.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_CSV}")

site_df = pd.read_csv(DATA_CSV)
relative_path = DATA_CSV.relative_to(PROJECT_DIR)

print(f"Loaded {len(site_df)} rows from {relative_path}")
site_df.head()

## Dataset integrity

Check the schema, sequence format, coordinates, and exact duplicate rows.

In [ ]:
required_columns = {
    "sequence",
    "label",
    "chrom",
    "strand",
    "position",
    "intron_start",
    "intron_end",
}
expected_labels = {"donor", "acceptor", "none"}
core_columns = ["sequence", "label", "chrom", "strand", "position"]
intron_columns = ["intron_start", "intron_end"]

missing_columns = required_columns.difference(site_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")

sequence_lengths = site_df["sequence"].str.len().value_counts().sort_index()
consistent_length = len(sequence_lengths) == 1
valid_alphabet = site_df["sequence"].str.fullmatch(r"[ACGT]+", na=False).all()
valid_labels = set(site_df["label"].dropna().unique()) == expected_labels
valid_strands = set(site_df["strand"].dropna().unique()).issubset({"+", "-"})
no_missing_core_values = not site_df[core_columns].isna().any().any()
positive_coordinates_present = not site_df.loc[
    site_df["label"] != "none", intron_columns
].isna().any().any()
negative_coordinates_missing = site_df.loc[
    site_df["label"] == "none", intron_columns
].isna().all().all()
exact_duplicate_rows = int(site_df.duplicated().sum())

checks = pd.DataFrame(
    {
        "result": [
            consistent_length,
            valid_alphabet,
            valid_labels,
            valid_strands,
            no_missing_core_values,
            positive_coordinates_present,
            negative_coordinates_missing,
            exact_duplicate_rows,
        ],
        "status": [
            "PASS" if consistent_length else "FAIL",
            "PASS" if valid_alphabet else "FAIL",
            "PASS" if valid_labels else "FAIL",
            "PASS" if valid_strands else "FAIL",
            "PASS" if no_missing_core_values else "FAIL",
            "PASS" if positive_coordinates_present else "FAIL",
            "PASS" if negative_coordinates_missing else "FAIL",
            "PASS" if exact_duplicate_rows == 0 else "REVIEW",
        ],
    },
    index=[
        "consistent sequence length",
        "A/C/G/T alphabet only",
        "expected labels present",
        "valid strand values",
        "no missing core values",
        "positive intron coordinates present",
        "negative intron coordinates missing",
        "exact duplicate rows",
    ],
)

display(checks)
print("\nSequence-length counts:")
print(sequence_lengths.to_string())

required_checks = [
    consistent_length,
    valid_alphabet,
    valid_labels,
    valid_strands,
    no_missing_core_values,
    positive_coordinates_present,
    negative_coordinates_missing,
]
if not all(required_checks):
    raise ValueError("A required integrity check failed")

sequence_length = int(sequence_lengths.index[0])
if sequence_length % 2 != 0:
    raise ValueError("Sequence length must be even")

window = sequence_length // 2
print(f"\nWindow: +/-{window} bases")

## Repeated sequence windows

Different genomic sites can share the same sequence. They are valid site records, but they are not independent modeling examples.

In [ ]:
site_df["sequence_group"], _ = pd.factorize(site_df["sequence"])

sequence_groups = (
    site_df.groupby("sequence_group")
    .agg(
        occurrences=("sequence", "size"),
        label_count=("label", "nunique"),
        labels=("label", lambda values: ", ".join(sorted(set(values)))),
        chromosome_count=("chrom", "nunique"),
        chromosomes=("chrom", lambda values: ", ".join(sorted(set(values)))),
    )
)

repeated_groups = sequence_groups.loc[
    sequence_groups["occurrences"] > 1
].sort_values("occurrences", ascending=False)
conflicting_groups = sequence_groups.loc[
    sequence_groups["label_count"] > 1
]

print(f"Repeated sequence groups: {len(repeated_groups)}")
print(f"Sequences with conflicting labels: {len(conflicting_groups)}")
display(repeated_groups)

if not conflicting_groups.empty:
    display(conflicting_groups)
    raise ValueError(
        "Identical sequences have conflicting labels and require review"
    )

# Keep the full site-level table unchanged and use one copy of each
# sequence-label combination for sequence-level EDA.
analysis_df = (
    site_df.drop_duplicates(subset=["sequence", "label"])
    .reset_index(drop=True)
    .copy()
)

print(f"All genomic sites: {len(site_df)}")
print(f"Unique sequence-label examples: {len(analysis_df)}")
print(f"Repeated examples excluded from EDA: {len(site_df) - len(analysis_df)}")

The remaining analyses use `analysis_df`, so repeated genomic copies do not receive extra weight. The original `site_df` is preserved for site-level provenance.

## Class balance

In [ ]:
site_counts = site_df["label"].value_counts()
analysis_counts = analysis_df["label"].value_counts()

class_balance = pd.DataFrame(
    {
        "all_sites": site_counts,
        "unique_examples": analysis_counts,
        "unique_percentage": (
            100 * analysis_counts / len(analysis_df)
        ).round(1),
    }
).fillna(0)

class_balance

## GC content

Large class differences could give a model an unintended shortcut.

In [ ]:
def gc_content(sequence):
    """
    Calculates the proportion of G and C bases in a DNA sequence.

    PARAMETERS
    ----------
    sequence : str
        DNA sequence containing A, C, G, and T

    RETURNS
    -------
    float
        proportion of bases that are G or C
    """
    return (sequence.count("G") + sequence.count("C")) / len(sequence)


analysis_df["gc_content"] = analysis_df["sequence"].apply(gc_content)
bins = np.linspace(
    analysis_df["gc_content"].min(),
    analysis_df["gc_content"].max(),
    31,
)

fig, ax = plt.subplots(figsize=(8, 5))
for label in ["donor", "acceptor", "none"]:
    values = analysis_df.loc[
        analysis_df["label"] == label,
        "gc_content",
    ]
    ax.hist(values, bins=bins, alpha=0.5, density=True, label=label)

ax.set(
    xlabel="GC content",
    ylabel="Density",
    title="GC content by class",
)
ax.legend()
fig.tight_layout()
plt.show()

## Positional base frequency

Position 0 is the first intronic base for donors and the final intronic base for acceptors.

In [ ]:
def positional_base_freq(sequences):
    """
    Calculates the frequency of each DNA base at every sequence position.

    PARAMETERS
    ----------
    sequences : list[str]
        equal-length DNA sequences

    RETURNS
    -------
    pandas.DataFrame
        positional frequencies for A, C, G, and T
    """
    if not sequences:
        raise ValueError("At least one sequence is required")

    length = len(sequences[0])
    frequencies = {base: np.zeros(length) for base in "ACGT"}

    for sequence in sequences:
        if len(sequence) != length:
            raise ValueError("All sequences must have the same length")
        for position, base in enumerate(sequence):
            frequencies[base][position] += 1

    return pd.DataFrame(
        {
            base: counts / len(sequences)
            for base, counts in frequencies.items()
        }
    )


relative_positions = np.arange(-window, window)
fig, axes = plt.subplots(3, 1, figsize=(10, 9), sharex=True)

for ax, label in zip(axes, ["donor", "acceptor", "none"]):
    sequences = analysis_df.loc[
        analysis_df["label"] == label,
        "sequence",
    ].tolist()
    frequencies = positional_base_freq(sequences)

    for base in "ACGT":
        ax.plot(relative_positions, frequencies[base], label=base)

    ax.axvline(0, color="gray", linestyle="--", linewidth=1)
    ax.set_title(f"{label} (n={len(sequences)})")
    ax.set_ylabel("Frequency")

axes[0].legend(loc="upper right")
axes[-1].set_xlabel("Position relative to splice-site anchor (nt)")
fig.tight_layout()
plt.show()

Correctly aligned donors should show a `GTATGT`-like pattern near position 0, while acceptors should show `AG` at positions −1 and 0. Negative sequences should not share a sharp positional motif.

## Exact splice-site motifs

Check donor `GT` and acceptor `AG` at their expected coordinates.

In [ ]:
donors = analysis_df.loc[analysis_df["label"] == "donor"].copy()
acceptors = analysis_df.loc[analysis_df["label"] == "acceptor"].copy()

donors["observed_motif"] = donors["sequence"].str.slice(
    window,
    window + 2,
)
acceptors["observed_motif"] = acceptors["sequence"].str.slice(
    window - 1,
    window + 1,
)

donor_matches = donors["observed_motif"].eq("GT")
acceptor_matches = acceptors["observed_motif"].eq("AG")

motif_summary = pd.DataFrame(
    {
        "expected": ["GT", "AG"],
        "matches": [donor_matches.sum(), acceptor_matches.sum()],
        "total": [len(donors), len(acceptors)],
        "percentage": [
            100 * donor_matches.mean(),
            100 * acceptor_matches.mean(),
        ],
    },
    index=["donor", "acceptor"],
).round({"percentage": 1})

donor_exceptions = donors.loc[~donor_matches].copy()
acceptor_exceptions = acceptors.loc[~acceptor_matches].copy()

donor_exceptions["context"] = donor_exceptions["sequence"].str.slice(
    window - 5,
    window + 7,
)
acceptor_exceptions["context"] = acceptor_exceptions["sequence"].str.slice(
    window - 6,
    window + 6,
)

exceptions = pd.concat(
    [
        donor_exceptions.assign(label="donor"),
        acceptor_exceptions.assign(label="acceptor"),
    ],
    ignore_index=True,
)

display(motif_summary)
print(f"Unique sites requiring review: {len(exceptions)}")
exceptions[
    ["label", "chrom", "strand", "position", "observed_motif", "context"]
]

## Before modeling

- Review the unique motif exceptions.
- Use `analysis_df` for an initial sequence-level model, or keep identical `sequence_group` values in the same split.
- Do not use `strand` as a predictor because every negative example currently has `+`.